# Combined notebook
This notebook contains all relevant code for the elec demand dataset.

In [1]:
import pandas as pd
import glob
import os

path = "../Dataset2_Demand/Raw/"
path_out = "../Dataset2_Demand/6_Elec_Demand_Final.csv"

all_files = glob.glob(os.path.join(path, "*.csv"))
dfs = []

for file in sorted(all_files):
    df = pd.read_csv(file)
    df.columns = df.columns.str.lower()

    year = os.path.basename(file).split('_')[-1].split('.')[0]
    df['year'] = int(year)
    if 'settlement_date' in df.columns:
        s = df['settlement_date'].astype(str).str.replace('"', '').str.strip()

        mask_mixed = s.str.match(r'^\d{2}-[A-Za-z]{3}-\d{4}$')

        dt_iso = pd.to_datetime(s.where(~mask_mixed), format="%Y-%m-%d", errors="coerce")

        dt_mixed = pd.to_datetime(s.where(mask_mixed), format="%d-%b-%Y", errors="coerce")

        df['settlement_date'] = dt_iso.fillna(dt_mixed).dt.strftime("%Y-%m-%d")

    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
df_subset = combined[(combined['year'] >= 2001) & (combined['year'] <= 2017)]

columns_to_drop = ['settlement_date', 'scottish_transfer', 'ifa_flow','nsl_flow', 'eleclink_flow', 'viking_flow', 'greenlink_flow', 'nd', 'tsd', 'non_bm_stor', 'pump_storage_pumping', 'scottish_transfer', 'ifa2_flow', 'britned_flow', 'moyle_flow', 'east_west_flow', 'nemo_flow', 'year']
df_filtered = df_subset.drop(columns=columns_to_drop, errors='ignore')

means = df_filtered.mean(numeric_only=True)
df_filled = df_filtered.fillna(means)

df_filled.to_csv(path_out, index=False)